In [1]:
import pandas as pd 
import numpy as np 
import re

In [2]:
raw=pd.read_excel("project raw file.xlsx", engine="openpyxl")
raw

,stNo,EmpName,surname
0,ST2462,Theodora Kansiime,Kansiime
1,ST1917,Oola Hilda,Oola
2,ST2208,Esebu Edward,Esebu
3,ST2735,Muwumba Norah,Muwumba
4,ST1787,Kababiito Winfred,Kababiito
...,...,...,...
340,ST2545,Shallot Atukunda,Atukunda
341,ST2419,Ileka Grace,Ileka
342,ST2446,Emuria Joseph,Emuria
343,ST1853,Mugisa Michael,Mugisa


In [3]:
import pandas as pd


# 1. CONFIGURE THIS FOR YOUR DATA

INPUT_FILE = "project raw file.xlsx"     
OUTPUT_FILE = "clean_output.xlsx"  

ID_COLUMN = "stNo"
NAME_COLUMN = "EmpName"

# 2. Surnames (and, where needed, a preferred first name) live HERE in the
# code, keyed by stNo — not in the spreadsheet. This means future files for
# these same people work even if they don't have a separate surname column.
#
# To add a new person: add a line "STxxxx": {"surname": "Whatever"},
# To fix a person whose sheet name is wrong/incomplete, also add
# "preferred_first_name": "Correct First Name" to their entry — that
# overrides whatever is in the spreadsheet for them.
name_rules = {
    "ST2462": {"surname": "Kansiime"},
    "ST1917": {"surname": "Oola"},
    "ST2208": {"surname": "Esebu"},
    "ST2735": {"surname": "Muwumba"},
    "ST1787": {"surname": "Kababiito"},
    "ST2451": {"surname": "Tumuhimbise"},
    "ST2585": {"surname": "Namagembe"},
    "ST1821": {"surname": "Kurawige"},
    "ST2754": {"surname": "Tumwesigye"},
    "ST1832": {"surname": "Lajja"},
    "ST1823": {"surname": "Kwarakunde"},
    "ST1594": {"surname": "Nassozi"},
    "ST1806": {"surname": "Kemigisha"},
    "ST2026": {"surname": "Kobusingye"},
    "ST2029": {"surname": "Kibwika"},
    "ST2144": {"surname": "Akii-Bua"},
    "ST2561": {"surname": "Muteesa"},
    "ST2318": {"surname": "Langol"},
    "ST2618": {"surname": "Kyakuwaire"},
    "ST2004": {"surname": "Obonyo"},
    "ST1589": {"surname": "Madaraka"},
    "ST2649": {"surname": "Andama"},
    "ST2751": {"surname": "Draru"},
    "ST2733": {"surname": "Kyarikunda"},
    "ST1833": {"surname": "Lifua"},
    "ST2207": {"surname": "Oferu"},
    "ST2572": {"surname": "Jjuuko"},
    "ST1721": {"surname": "Mugisha"},
    "ST2236": {"surname": "Isabirye"},
    "ST1889": {"surname": "Nambi"},
    "ST2180": {"surname": "Uwera"},
    "ST1753": {"surname": "Ashaba-Aheebwa"},
    "ST1971": {"surname": "Kantono"},
    "ST1638": {"surname": "Nalumansi"},
    "ST1775": {"surname": "Busuulwa"},
    "ST2064": {"surname": "Kasirye"},
    "ST2174": {"surname": "Kansiime"},
    "ST1871": {"surname": "Nabbona"},
    "ST1864": {"surname": "Mutono"},
    "ST1918": {"surname": "Otim"},
    "ST1590": {"surname": "Mbabazi"},
    "ST1907": {"surname": "Nyangoma"},
    "ST1975": {"surname": "Busingye"},
    "ST2017": {"surname": "Munyambabazi"},
    "ST2065": {"surname": "Aleesi"},
    "ST2141": {"surname": "Woniaye"},
    "ST1804": {"surname": "Kawenyera"},
    "ST2388": {"surname": "Gwayambadde"},
    "ST2346": {"surname": "Nakimbugwe"},
    "ST1894": {"surname": "Nankunda"},
    "ST2424": {"surname": "Kyomuhendo"},
    "ST1928": {"surname": "Ssonko"},
    "ST2578": {"surname": "Kirangwa"},
    "ST2579": {"surname": "Nangendo"},
    "ST1867": {"surname": "Mwebembezi"},
    "ST2033": {"surname": "Byarugaba"},
    "ST2417": {"surname": "Baranga"},
    "ST2634": {"surname": "Vumya"},
    "ST1845": {"surname": "Mirembe"},
    "ST1836": {"surname": "Magara"},
    "ST1931": {"surname": "Tibewanwa"},
    "ST1898": {"surname": "Nantaba"},
    "ST1614": {"surname": "Kayemba"},
    "ST1990": {"surname": "Birungi"},
    "ST2020": {"surname": "Ahumwire"},
    "ST1979": {"surname": "Oyesigye"},
    "ST2028": {"surname": "Muhangi"},
    "ST1846": {"surname": "Mpairwe"},
    "ST1938": {"surname": "Twehamye"},
    "ST2556": {"surname": "Rukundo"},
    "ST2632": {"surname": "Ngasirwa"},
    "ST2132": {"surname": "Najjingo"},
    "ST2441": {"surname": "Nabbosa"},
    "ST2614": {"surname": "Kiiza"},
    "ST2616": {"surname": "Kanyike"},
    "ST2007": {"surname": "Kwanya"},
    "ST2398": {"surname": "Bangirana"},
    "ST1840": {"surname": "Mawanda"},
    "ST1625": {"surname": "Karara"},
    "ST1745": {"surname": "Akurut"},
    "ST2091": {"surname": "Ayebazibwe"},
    "ST2105": {"surname": "Nakabuye"},
    "ST2175": {"surname": "Wagabaza"},
    "ST2548": {"surname": "Arinda"},
    "ST2336": {"surname": "Kirunda"},
    "ST2397": {"surname": "Taremwa"},
    "ST1759": {"surname": "Atuheire"},
    "ST2378": {"surname": "Mugume"},
    "ST2367": {"surname": "Kauma"},
    "ST2701": {"surname": "Ndagire"},
    "ST1965": {"surname": "Warugaba"},
    "ST2558": {"surname": "Kasangaki"},
    "ST2562": {"surname": "Nakitende"},
    "ST2746": {"surname": "Wabwire"},
    "ST2762": {"surname": "Totto"},
    "ST2654": {"surname": "Kayongo"},
    "ST2122": {"surname": "Kyomugisha"},
    "ST2494": {"surname": "Awino"},
    "ST2497": {"surname": "Nabacwa"},
    "ST2612": {"surname": "Nandawula"},
    "ST2444": {"surname": "Amongi"},
    "ST1803": {"surname": "Kaweesi"},
    "ST2024": {"surname": "Kobwemi"},
    "ST2739": {"surname": "Tumwine"},
    "ST2644": {"surname": "Ariho"},
    "ST2699": {"surname": "Kyarukunda"},
    "ST2717": {"surname": "Babwetera"},
    "ST2745": {"surname": "Nshemereirwe"},
    "ST2766": {"surname": "Munywevu"},
    "ST2772": {"surname": "Azarius"},
    "ST1749": {"surname": "Amwesiga"},
    "ST2350": {"surname": "Tumwesigye"},
    "ST1934": {"surname": "Tumwebaze"},
    "ST2395": {"surname": "Doka"},
    "ST2408": {"surname": "Kamuli"},
    "ST2567": {"surname": "Angucia"},
    "ST2750": {"surname": "Acen"},
    "ST2615": {"surname": "Mulinzi"},
    "ST2732": {"surname": "Zawedde"},
    "ST2771": {"surname": "Odur"},
    "ST2385": {"surname": "Ayikoru"},
    "ST2454": {"surname": "Atwine"},
    "ST1919": {"surname": "Rukundo"},
    "ST2376": {"surname": "Opiru"},
    "ST1947": {"surname": "Bogere"},
    "ST2389": {"surname": "Kemigisa"},
    "ST2752": {"surname": "Nakkazi"},
    "ST2457": {"surname": "Kiiza"},
    "ST2213": {"surname": "Ampaire"},
    "ST1472": {"surname": "Amoni"},
    "ST2727": {"surname": "Nagaba"},
    "ST2728": {"surname": "Mbabazi"},
    "ST2658": {"surname": "Muruubya"},
    "ST2364": {"surname": "Okitoi"},
    "ST2440": {"surname": "Serwanga"},
    "ST2472": {"surname": "Orono"},
    "ST2504": {"surname": "Kizza"},
    "ST2506": {"surname": "Atenge"},
    "ST2533": {"surname": "Nakato"},
    "ST1878": {"surname": "Nagujja"},
    "ST1929": {"surname": "Suubi"},
    "ST1915": {"surname": "Olupot"},
    "ST1449": {"surname": "Namusubo"},
    "ST1770": {"surname": "Birabwa"},
    "ST1756": {"surname": "Asiimwe"},
    "ST1747": {"surname": "Alicabiaku"},
    "ST1744": {"surname": "Ituka"},
    "ST1891": {"surname": "Namukoli"},
    "ST2000": {"surname": "Nabaasa"},
    "ST2039": {"surname": "Nankya"},
    "ST2046": {"surname": "Nankya"},
    "ST2176": {"surname": "Asinai"},
    "ST2311": {"surname": "Luzinda"},
    "ST2565": {"surname": "Zawedde"},
    "ST2603": {"surname": "Aloti"},
    "ST2737": {"surname": "Mpumwire"},
    "ST1964": {"surname": "Atuheire"},
    "ST2155": {"surname": "Langa"},
    "ST2347": {"surname": "Akello"},
    "ST2617": {"surname": "Nalubega"},
    "ST2557": {"surname": "Owomuhangi"},
    "ST2363": {"surname": "Kyohirwe"},
    "ST2416": {"surname": "Komuhendo"},
    "ST2491": {"surname": "Aripa"},
    "ST2492": {"surname": "Mukunzi"},
    "ST2493": {"surname": "Atuhaire"},
    "ST2495": {"surname": "Muhoozi"},
    "ST2508": {"surname": "Kalanguka"},
    "ST2509": {"surname": "Kagwisa"},
    "ST2527": {"surname": "Kisaakye"},
    "ST2528": {"surname": "Naluyange"},
    "ST1850": {"surname": "Mudondo"},
    "ST1899": {"surname": "Nanteza"},
    "ST1893": {"surname": "Namwima"},
    "ST2664": {"surname": "Kalanda"},
    "ST1709": {"surname": "Nabulime"},
    "ST1994": {"surname": "Busingye"},
    "ST2089": {"surname": "Namara"},
    "ST2113": {"surname": "Natukunda"},
    "ST2391": {"surname": "Lubega"},
    "ST2496": {"surname": "Ainembabazi"},
    "ST2514": {"surname": "Nansamba"},
    "ST2582": {"surname": "Onyait"},
    "ST2584": {"surname": "Alemu"},
    "ST2607": {"surname": "Sawula"},
    "ST2613": {"surname": "Mbabazi"},
    "ST1869": {"surname": "Nabaasa"},
    "ST1923": {"surname": "Sembatya"},
    "ST2299": {"surname": "Ssenkungu"},
    "ST2453": {"surname": "Wahumura"},
    "ST2464": {"surname": "Nalubowa"},
    "ST2477": {"surname": "Nakiyemba"},
    "ST2631": {"surname": "Nakiwala"},
    "ST1847": {"surname": "Mpalaganyi"},
    "ST1834": {"surname": "Kakumba"},
    "ST1895": {"surname": "Nansamba"},
    "ST2030": {"surname": "Kasajja"},
    "ST2075": {"surname": "Mulondo"},
    "ST2559": {"surname": "Ayebare"},
    "ST2571": {"surname": "Sanyu"},
    "ST2339": {"surname": "Charity"},
    "ST2501": {"surname": "Musaazi"},
    "ST2619": {"surname": "Kyomuhendo"},
    "ST1765": {"surname": "Babita"},
    "ST1615": {"surname": "Namutosi"},
    "ST2131": {"surname": "Babirye"},
    "ST2402": {"surname": "Owomugisha"},
    "ST1880": {"surname": "Nakaziba"},
    "ST2470": {"surname": "Akurut"},
    "ST1865": {"surname": "Muwanse"},
    "ST1968": {"surname": "Namatovu"},
    "ST1666": {"surname": "Ssemakula"},
    "ST2610": {"surname": "Asiimwe"},
    "ST1742": {"surname": "Akimanzi"},
    "ST1877": {"surname": "Nagadya"},
    "ST2016": {"surname": "Sajjabi"},
    "ST2695": {"surname": "Otim"},
    "ST2758": {"surname": "Opiyo"},
    "ST2452": {"surname": "Fazira"},
    "ST2568": {"surname": "Nabirye"},
    "ST1728": {"surname": "Waiswa"},
    "ST2563": {"surname": "Bagire"},
    "ST2606": {"surname": "Ndifuna"},
    "ST1872": {"surname": "Nabbosa"},
    "ST1857": {"surname": "Muhangi"},
    "ST2373": {"surname": "Ongom"},
    "ST2645": {"surname": "Muganza"},
    "ST1797": {"surname": "Kanyonyi"},
    "ST2302": {"surname": "Kizito"},
    "ST1660": {"surname": "Andama"},
    "ST1598": {"surname": "Oryonga"},
    "ST2741": {"surname": "Okirio"},
    "ST2500": {"surname": "Namanya"},
    "ST2554": {"surname": "Arinaitwe"},
    "ST2753": {"surname": "Karugaba"},
    "ST1761": {"surname": "Atwijukire"},
    "ST2112": {"surname": "Aine-Amaani"},
    "ST2691": {"surname": "Kigundu"},
    "ST1913": {"surname": "Okiror"},
    "ST2608": {"surname": "Eriu"},
    "ST2574": {"surname": "Opio"},
    "ST2549": {"surname": "Okiror"},
    "ST1910": {"surname": "Ogwang"},
    "ST2239": {"surname": "Luzzi"},
    "ST2773": {"surname": "Komugisha"},
    "ST2635": {"surname": "Amulen"},
    "ST2731": {"surname": "Mwebaze"},
    "ST2522": {"surname": "Turyahabwe"},
    "ST1937": {"surname": "Twahirwa"},
    "ST2742": {"surname": "Ecweu"},
    "ST2765": {"surname": "Nduhuura"},
    "ST2734": {"surname": "Namaganda"},
    "ST1794": {"surname": "Kairagura"},
    "ST1584": {"surname": "Aheebwa"},
    "ST2359": {"surname": "Egonu"},
    "ST2714": {"surname": "Kassim"},
    "ST2534": {"surname": "Nalule"},
    "ST2552": {"surname": "Wegosasa"},
    "ST2768": {"surname": "Wangi"},
    "ST2625": {"surname": "Fatuma"},
    "ST1841": {"surname": "Mayanja"},
    "ST1712": {"surname": "Habiyaremye"},
    "ST2357": {"surname": "Bunkeddeko"},
    "ST2027": {"surname": "Kisembo"},
    "ST2551": {"surname": "Agaba"},
    "ST2730": {"surname": "Nalunkuuma"},
    "ST2097": {"surname": "Nyiraguhirwa"},
    "ST2755": {"surname": "Kirangwa"},
    "ST2447": {"surname": "Kanavita"},
    "ST2604": {"surname": "Musiimenta"},
    "ST2560": {"surname": "Matsiko"},
    "ST1636": {"surname": "Nkurunziza"},
    "ST1734": {"surname": "Ahabwe"},
    "ST1671": {"surname": "Mwesigwa"},
    "ST1366": {"surname": "Dadye"},
    "ST1509": {"surname": "Kiiza"},
    "ST2463": {"surname": "Kajuma"},
    "ST2401": {"surname": "Ocen"},
    "ST2555": {"surname": "Namaganda"},
    "ST2611": {"surname": "Otim"},
    "ST1868": {"surname": "Mwesigye"},
    "ST2035": {"surname": "Tugume"},
    "ST2370": {"surname": "Nangobi"},
    "ST1859": {"surname": "Mukoyonzo"},
    "ST1588": {"surname": "Kalema"},
    "ST2045": {"surname": "Wandera"},
    "ST1619": {"surname": "Walangaire"},
    "ST2659": {"surname": "Mulyagonja"},
    "ST2448": {"surname": "Nabagereka"},
    "ST2458": {"surname": "Wasswa"},
    "ST2586": {"surname": "Atenia"},
    "ST1817": {"surname": "Kobusinge"},
    "ST2661": {"surname": "Nuwagaba"},
    "ST2697": {"surname": "Kasujja"},
    "ST2702": {"surname": "Mbaziira"},
    "ST2703": {"surname": "Kebirungi"},
    "ST2704": {"surname": "Atugonza"},
    "ST2705": {"surname": "Nalule"},
    "ST2706": {"surname": "Tusiimire"},
    "ST2707": {"surname": "Namale"},
    "ST2774": {"surname": "Obbo"},
    "ST2086": {"surname": "Aliisa"},
    "ST2186": {"surname": "Tugume"},
    "ST2550": {"surname": "Aharizira"},
    "ST2763": {"surname": "Kwikiriza"},
    "ST2764": {"surname": "Natukunda"},
    "ST2219": {"surname": "Kyomuhendo"},
    "ST2222": {"surname": "Muhigwa"},
    "ST2344": {"surname": "Naggayi"},
    "ST2345": {"surname": "Nalwoga"},
    "ST2524": {"surname": "Nalubega"},
    "ST1740": {"surname": "Akedi"},
    "ST2002": {"surname": "Namirimu"},
    "ST2140": {"surname": "Ankunda"},
    "ST2466": {"surname": "Abdulhakim"},
    "ST2576": {"surname": "Omona"},
    "ST2795": {"surname": "Gonzaga"},
    "ST2696": {"surname": "Komugisa"},
    "ST2698": {"surname": "Santina"},
    "ST2172": {"surname": "Achou"},
    "ST2744": {"surname": "Nerima"},
    "ST2133": {"surname": "Alemukori"},
    "ST2757": {"surname": "Ojoo"},
    "ST2525": {"surname": "Nagujja"},
    "ST2321": {"surname": "Imuaan"},
    "ST2708": {"surname": "Mabonga"},
    "ST2564": {"surname": "Nyakecho"},
    "ST2499": {"surname": "Kasirye"},
    "ST2660": {"surname": "Owori"},
    "ST1959": {"surname": "Rwanyaga"},
    "ST2627": {"surname": "Kharono"},
    "ST2340": {"surname": "Byamukama"},
    "ST2502": {"surname": "Ichumar"},
    "ST1881": {"surname": "Nakigudde"},
    "ST1820": {"surname": "Komugisha"},
    "ST1921": {"surname": "Sajjabi"},
    "ST1888": {"surname": "Namayanja"},
    "ST1980": {"surname": "Wanyama"},
    "ST2057": {"surname": "Muyinza"},
    "ST2129": {"surname": "Kakyama"},
    "ST2545": {"surname": "Atukunda"},
    "ST2419": {"surname": "Ileka"},
    "ST2446": {"surname": "Emuria"},
    "ST1853": {"surname": "Mugisa"},
    "ST1942": {"surname": "Atuhairwe"},
    # Example of an override for a name that's wrong/incomplete in the sheet:
    # "ST9999": {"surname": "Sabiiti", "preferred_first_name": "Phiona"},
}


# 3. SCRIPT LOGIC 

def standardize_one_name(full_name: str, st_no, rules: dict):
    """
    Return a standardized name for one row.
    The surname is forced to be the last part of the name.
    If a preferred first name is supplied for this ID, that is used instead
    of whatever first name is in the spreadsheet.
    """
    rule = rules.get(str(st_no).strip())
    if not rule:
        # No entry for this ID -> we don't know the surname, leave as-is.
        return full_name

    if not isinstance(full_name, str) or not full_name.strip():
        full_name = ""

    surname = str(rule.get("surname", "")).strip()
    preferred_first_name = str(rule.get("preferred_first_name", "")).strip()

    if not surname:
        return full_name

    words = [w for w in full_name.split() if w]
    surname_lower = surname.lower()

    # If a preferred first name is supplied, it overrides the sheet entirely.
    if preferred_first_name:
        return f"{preferred_first_name.title()} {surname.title()}"

    if not words:
        return full_name

    if words[-1].lower() == surname_lower:
        # Already last -> leave it alone (matches the original casing).
        return " ".join(words)

    if surname_lower in [w.lower() for w in words]:
        # Move the surname to the end, keep everything else in order.
        others = [w for w in words if w.lower() != surname_lower]
        return " ".join(others + [surname])

    # Surname for this ID doesn't appear anywhere in the sheet's name ->
    # leave untouched rather than guess.
    return full_name


raw = pd.read_excel(INPUT_FILE, dtype=str)
raw[NAME_COLUMN] = raw[NAME_COLUMN].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

raw[NAME_COLUMN] = [
    standardize_one_name(row[NAME_COLUMN], row[ID_COLUMN], name_rules)
    for _, row in raw.iterrows()
]

result = raw[[ID_COLUMN, NAME_COLUMN]].copy()
result.to_excel(OUTPUT_FILE, index=False)
print(f"Done. {len(result)} rows written to {OUTPUT_FILE}")



Done. 345 rows written to clean_output.xlsx


In [4]:
clean= pd.read_excel("clean_output.xlsx", engine="openpyxl")
clean

,stNo,EmpName
0,ST2462,Theodora Kansiime
1,ST1917,Hilda Oola
2,ST2208,Edward Esebu
3,ST2735,Norah Muwumba
4,ST1787,Winfred Kababiito
...,...,...
340,ST2545,Shallot Atukunda
341,ST2419,Grace Ileka
342,ST2446,Joseph Emuria
343,ST1853,Michael Mugisa


In [ ]:
# Money cleaning function (for future files)
# def standardize_money(value):
#     if pd.isna(value):
#         return value

#     text = str(value).lower().replace(',', '').strip()
#     text = text.replace('ugx', '').strip()

#     multipliers = {
#         'k': 1_000,
#         'thousand': 1_000,
#         'm': 1_000_000,
#         'million': 1_000_000,
#         'b': 1_000_000_000,
#         'billion': 1_000_000_000,
#     }

#     match = re.match(r'(\d+(?:\.\d+)?)\s*([a-z]+)?', text)
#     if match:
#         num = float(match.group(1))
#         suffix = match.group(2)

#         if suffix in multipliers:
#             num *= multipliers[suffix]

#         return int(num)

#     return value